# Inspect raw Dota replay data

Use this notebook to understand the raw tables and investigate possible data-quality issues.
It reads local Parquet files with **pandas and PyArrow**, and does not apply cleaning rules or
write outputs. The [cleaning walkthrough](cleaning_walkthrough.ipynb) explains fixes and checks
whether they work. Raw nulls, repeated keys, and unusual values are observations, not automatic errors.

Install dependencies with `python -m pip install -r notebooks/requirements.txt` from the repository
root and select that Python environment in VS Code or Jupyter. Run the cells in order.

Workflow: **check coverage → understand tables → preview rows → profile quality → investigate**.
References: [raw contract](../docs/data-contract.md), [extraction spec](../spark_jobs/replay_spec.py).

In [80]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "spark_jobs" / "replay_spec.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook from the dota-pipeline repository.")
PARSED = ROOT / "output" / "parsed"
FOCUS_MATCH_ID = None  # Set a completed match ID; None uses the first available match.
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Meaning of one raw row, and columns to profile for repeated raw keys.
# Combat events have no unique event ID; a repeated tick is not itself an error.
TABLES = {
    "match": ("One match header", []),
    "players": ("One player in the match header", ["slot"]),
    "combatlog": ("One combat-log event; fields depend on event type", None),
    "names": ("One combat-log name lookup entry", ["idx"]),
    "entity_names": ("One entity-name lookup entry", ["idx"]),
    "gamerules": ("One game-rules entity at a sampled tick", ["tick", "handle"]),
    "heroes": ("One hero entity at a sampled tick; illusions included", ["tick", "handle"]),
    "player_resource": ("One player-resource array entry at a sampled tick", ["tick", "handle", "index"]),
    "team_data": ("One team-data array entry at a sampled tick", ["tick", "handle", "index"]),
    "wards": ("One ward entity creation or deletion", ["tick", "event", "handle"]),
    "item_entities": ("One item entity creation or deletion", ["tick", "event", "handle"]),
}

def read_table(table, match_id):
    files = sorted((PARSED / table / f"match_id={match_id}").glob("*.parquet"))
    if not files:
        return None
    df = pd.concat([pq.ParquetFile(f).read().to_pandas().convert_dtypes() for f in files], ignore_index=True)
    if "match_id" in df and not df["match_id"].eq(match_id).all():
        raise ValueError(f"Embedded match ID disagrees with partition: {table}/{match_id}")
    # Recover partition metadata; do not change any replay property values.
    df["match_id"] = pd.Series(match_id, index=df.index, dtype="Int64")
    return df

## 1. Which matches and tables are available?

Completion markers identify parsed matches even when the original `.dem` files have been moved.
Local replay files without a marker are listed as pending. A missing wards partition is allowed;
other missing tables are flagged below rather than silently treated as valid empty data.
Finish parsing before inspecting a match so its files are not being overwritten during reading.

In [81]:
replay_ids = {int(p.stem) for p in (ROOT / "data").glob("*.dem") if p.stem.isdigit()}
match_ids = sorted(int(p.name.split("=", 1)[1]) for p in (PARSED / "_complete").glob("match_id=*")
    if p.name.split("=", 1)[1].isdigit() and any(p.glob("*.parquet")))
print("Completed matches:", match_ids)
print("Local replays still awaiting parsing:", sorted(replay_ids - set(match_ids)))
if not match_ids:
    raise RuntimeError(f"No completed parsed matches found in {PARSED}.")
MATCH_ID = FOCUS_MATCH_ID if FOCUS_MATCH_ID is not None else match_ids[0]
if MATCH_ID not in match_ids:
    raise ValueError("FOCUS_MATCH_ID must be one of the completed matches above.")
print("Selected match:", MATCH_ID)

raw = {}; coverage_rows = []; schema_rows = []
for table in TABLES:
    parts = []
    for match_id in match_ids:
        df = read_table(table, match_id)
        coverage_rows.append({"match_id": match_id, "table": table,
            "rows": 0 if df is None else len(df), "partition_present": df is not None,
            "missing_required_table": df is None and table != "wards"})
        if df is not None:
            parts.append(df)
            schema_rows.extend({"table": table, "match_id": match_id, "column": col, "dtype": str(dtype)}
                               for col, dtype in df.dtypes.items())
    if parts:
        raw[table] = pd.concat(parts, ignore_index=True)
coverage = pd.DataFrame(coverage_rows)
schemas = pd.DataFrame(schema_rows)
display(coverage.pivot(index="match_id", columns="table", values="rows"))
print("Missing required partitions:")
display(coverage.loc[coverage["missing_required_table"]])

Completed matches: [9079737909, 9079737994, 9079738443, 9079738574, 9079740225, 9079740632, 9079741152, 9079743269, 9079745340, 9079746862]
Local replays still awaiting parsing: []
Selected match: 9079737909


table,combatlog,entity_names,gamerules,heroes,item_entities,match,names,player_resource,players,team_data,wards
match_id,,,,,,,,,,,
9079737909,21908,564,1508,14570,621,1,242,15080,10,15080,0
9079737994,31712,607,1348,14238,608,1,298,13480,10,13480,6
9079738443,14679,547,918,8154,342,1,218,9180,10,9180,10
9079738574,23895,605,1379,17525,1173,1,336,13790,10,13790,57
9079740225,6132,536,545,4320,185,1,155,5450,10,5450,5
9079740632,19802,579,1503,15356,439,1,271,15030,10,15030,6
9079741152,14863,581,1199,10740,443,1,291,12723,10,11990,17
9079743269,6613,555,761,24360,302,1,200,7610,10,7610,14
9079745340,10265,557,1079,10551,557,1,241,10790,10,10790,34


Missing required partitions:


,match_id,table,rows,partition_present,missing_required_table


## 2. What does each table contain?

The catalog describes row meaning, row counts, and column counts. Select a table to inspect its
column types across matches. Multiple observed dtypes are a prompt to inspect the source values;
for example, an all-null column can infer a different pandas dtype in one match.

In [82]:
catalog = pd.DataFrame([{"table": table, "one_row_means": description,
    "total_rows": len(raw[table]) if table in raw else 0,
    "columns": len(raw[table].columns) if table in raw else 0}
    for table, (description, _) in TABLES.items()])
display(catalog)
TABLE_TO_DESCRIBE = "player_resource"
if TABLE_TO_DESCRIBE not in TABLES:
    raise ValueError("Choose a table from the catalog.")
selected_schema = schemas.loc[schemas["table"].eq(TABLE_TO_DESCRIBE)]
column_types = selected_schema.groupby("column").agg(
    observed_dtypes=("dtype", lambda s: ", ".join(sorted(set(s)))), matches=("match_id", "nunique"))
display(column_types)

,table,one_row_means,total_rows,columns
0,match,One match header,10,4
1,players,One player in the match header,100,4
2,combatlog,One combat-log event; fields depend on event type,157132,80
3,names,One combat-log name lookup entry,2457,3
4,entity_names,One entity-name lookup entry,5703,3
5,gamerules,One game-rules entity at a sampled tick,11300,9
6,heroes,One hero entity at a sampled tick; illusions included,129198,28
7,player_resource,One player-resource array entry at a sampled tick,113733,12
8,team_data,One team-data array entry at a sampled tick,113000,22
9,wards,One ward entity creation or deletion,174,11


,observed_dtypes,matches
column,,
class,string,10
handle,Int32,10
index,Int32,10
m_vecPlayerData.m_iPlayerTeam,Int32,10
m_vecPlayerTeamData.m_iAssists,"Int32, Int64",10
m_vecPlayerTeamData.m_iDeaths,"Int32, Int64",10
m_vecPlayerTeamData.m_iKills,"Int32, Int64",10
m_vecPlayerTeamData.m_iLevel,"Int32, Int64",10
m_vecPlayerTeamData.m_iTeamSlot,"Int32, Int64",10


## 3. Show the first five rows of every table for one match

Set `FOCUS_MATCH_ID` in setup and rerun to change the selected match. Rows are ordered by raw
tick where available. No indexes are resolved, players remapped, or rows removed.

In [87]:
for table in TABLES:
    print(f"\n{table} | match {MATCH_ID}")
    if table not in raw:
        print("No partition available")
        continue
    sample = raw[table].loc[raw[table]["match_id"].eq(MATCH_ID)]
    if sample.empty:
        print("No rows for this match")
    else:
        display((sample.sort_values("tick", kind="stable") if "tick" in sample else sample).head(5))
        x = len(sample)
        print("Number of rows: " + str(x))


match | match 9079737909


,game_mode,game_winner,end_time,match_id
0,22,3,1794045663,9079737909


Number of rows: 1

players | match 9079737909


,slot,hero_name,game_team,match_id
0,0,npc_dota_hero_juggernaut,2,9079737909
1,1,npc_dota_hero_legion_commander,2,9079737909
2,2,npc_dota_hero_tidehunter,2,9079737909
3,3,npc_dota_hero_axe,2,9079737909
4,4,npc_dota_hero_sniper,2,9079737909


Number of rows: 10

combatlog | match 9079737909


,tick,type,timestamp,attacker_name_idx,target_name_idx,inflictor_name_idx,value,ability_level,ability_toggle_off,ability_toggle_on,armor_debuff_modifier,assist_players,at_night_time,attacker_has_scepter,attacker_hero,...,target_self,target_source_name_idx,target_team,timestamp_raw,total_unit_death_count,tracked_stat_id,ultimate_ability,unit_status_label,uses_charges,visible_dire,visible_radiant,will_reincarnate,xp_reason,xpm,match_id
0,2,DOTA_COMBATLOG_GAME_STATE,13.433334,<NA>,<NA>,<NA>,2,<NA>,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,13.433334,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,9079737909
1,2,DOTA_COMBATLOG_GAME_STATE,13.433334,<NA>,<NA>,<NA>,12,<NA>,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,13.433334,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,9079737909
2,2,DOTA_COMBATLOG_GAME_STATE,13.433334,<NA>,<NA>,<NA>,9,<NA>,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,13.433334,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,9079737909
3,158,DOTA_COMBATLOG_GAME_STATE,18.633335,<NA>,<NA>,<NA>,3,<NA>,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,18.633335,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,9079737909
4,1058,DOTA_COMBATLOG_GAME_STATE,48.666668,<NA>,<NA>,<NA>,8,<NA>,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,48.666668,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,9079737909


... (11 outputs left out)

,tick,handle,class,index,m_vecDataTeam.m_iNetWorth,m_vecDataTeam.m_iReliableGold,m_vecDataTeam.m_iUnreliableGold,m_vecDataTeam.m_iTotalEarnedGold,m_vecDataTeam.m_iTotalEarnedXP,m_vecDataTeam.m_iLastHitCount,m_vecDataTeam.m_iDenyCount,m_vecDataTeam.m_flHeroDamage,m_vecDataTeam.m_flTowerDamage,m_vecDataTeam.m_fHealing,m_vecDataTeam.m_fStuns,m_vecDataTeam.m_iObserverWardsPlaced,m_vecDataTeam.m_iSentryWardsPlaced,m_vecDataTeam.m_iCampsStacked,m_vecDataTeam.m_iRunePickups,m_vecDataTeam.m_iTowerKills,m_vecDataTeam.m_iRoshanKills,match_id
0,30,3375246,CDOTA_DataRadiant,0,0,0,600,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,9079737909
1,30,3375246,CDOTA_DataRadiant,1,0,0,600,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,9079737909
2,30,3375246,CDOTA_DataRadiant,2,0,0,600,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,9079737909
3,30,3375246,CDOTA_DataRadiant,3,0,0,600,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,9079737909
4,30,3375246,CDOTA_DataRadiant,4,0,0,600,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,9079737909


Number of rows: 15080

wards | match 9079737909
No rows for this match

item_entities | match 9079737909


,tick,event,handle,class,m_pEntity.m_nameStringTableIndex,match_id
0,14,created,15352007,CDOTA_Item_TeleportScroll,310,9079737909
1,14,created,14926024,CDOTA_Item_TeleportScroll,310,9079737909
2,16,created,3293385,CDOTA_Item_TeleportScroll,310,9079737909
3,16,created,3735754,CDOTA_Item_TeleportScroll,310,9079737909
4,78,created,5488843,CDOTA_Item_TeleportScroll,310,9079737909


Number of rows: 621


## 4. Missing values and repeated raw keys

The quality summary reports each table and match separately. `exact_repeated_rows` counts copies
after the first identical full row. `repeated_key_groups` counts groups sharing the profiled raw
key; it does not decide whether to delete or aggregate anything. Combat events lack a unique raw
event ID, so their key count is left blank. View event-specific sparsity separately below.

In [ ]:
quality_rows = []
null_rows = []
for table, df in raw.items():
    key = TABLES[table][1]
    for match_id, part in df.groupby("match_id", dropna=False):
        keys = ["match_id", *key] if key is not None else None
        counts = part.groupby(keys, dropna=False).size() if keys is not None else None
        quality_rows.append({"match_id": match_id, "table": table, "rows": len(part),
            "rows_with_any_null": int(part.isna().any(axis=1).sum()),
            "exact_repeated_rows": int(part.duplicated().sum()),
            "null_key_rows": int(part[keys].isna().any(axis=1).sum()) if keys is not None else pd.NA,
            "repeated_key_groups": int(counts.gt(1).sum()) if counts is not None else pd.NA})
        null_rows.extend({"match_id": match_id, "table": table, "column": col,
            "nulls": int(n), "null_percent": round(100 * n / len(part), 2) if len(part) else 0}
            for col, n in part.isna().sum().items())
quality = pd.DataFrame(quality_rows)
null_profile = pd.DataFrame(null_rows)
display(quality)
print("Repeated rows, repeated keys, or null key fields:")
display(quality.loc[(quality["exact_repeated_rows"] > 0)
    | quality["repeated_key_groups"].gt(0).fillna(False) | quality["null_key_rows"].gt(0).fillna(False)])
NULL_TABLE = "combatlog"
NULL_MATCH_ID = MATCH_ID
print("Columns containing nulls for selected table/match:")
display(null_profile.loc[null_profile["table"].eq(NULL_TABLE)
    & null_profile["match_id"].eq(NULL_MATCH_ID) & null_profile["nulls"].gt(0)].sort_values("nulls", ascending=False))

,match_id,table,rows,rows_with_any_null,exact_repeated_rows,null_key_rows,repeated_key_groups
0,9079737909,match,1,0,0,0,0
1,9079737994,match,1,0,0,0,0
2,9079738443,match,1,0,0,0,0
3,9079738574,match,1,0,0,0,0
4,9079740225,match,1,0,0,0,0
...,...,...,...,...,...,...,...
104,9079740632,item_entities,439,0,0,0,0
105,9079741152,item_entities,443,0,0,0,0
106,9079743269,item_entities,302,0,0,0,0
107,9079745340,item_entities,557,0,0,0,0


Repeated rows, repeated keys, or null key fields:


,match_id,table,rows,rows_with_any_null,exact_repeated_rows,null_key_rows,repeated_key_groups
20,9079737909,combatlog,21908,21908,101,<NA>,<NA>
21,9079737994,combatlog,31712,31712,3334,<NA>,<NA>
22,9079738443,combatlog,14679,14679,68,<NA>,<NA>
23,9079738574,combatlog,23895,23895,1922,<NA>,<NA>
24,9079740225,combatlog,6132,6132,8,<NA>,<NA>
25,9079740632,combatlog,19802,19802,149,<NA>,<NA>
26,9079741152,combatlog,14863,14863,387,<NA>,<NA>
27,9079743269,combatlog,6613,6613,131,<NA>,<NA>
28,9079745340,combatlog,10265,10265,62,<NA>,<NA>
29,9079746862,combatlog,7263,7263,38,<NA>,<NA>


Columns containing nulls for selected table/match:


,match_id,table,column,nulls,null_percent
158,9079737909,combatlog,xpm,21908,100.00
95,9079737909,combatlog,attacker_hero_level,21908,100.00
113,9079737909,combatlog,kill_eater_event,21908,100.00
112,9079737909,combatlog,invisibility_modifier,21908,100.00
111,9079737909,combatlog,inflictor_is_stolen_ability,21908,100.00
...,...,...,...,...,...
142,9079737909,combatlog,target_hero,2883,13.16
144,9079737909,combatlog,target_illusion,2883,13.16
145,9079737909,combatlog,target_self,2591,11.83
86,9079737909,combatlog,value,531,2.42


## 5. Combat event counts and missing fields by event type

Fields can be absent because they do not apply to the event. For example, a missing attacker on
a purchase is different from a missing damage amount on a damage event. `rows` counts all raw
events before hero/illusion filters. This table reports observations and does not fill nulls.

In [ ]:
combat = raw["combatlog"]
profile_fields = ["attacker_name_idx", "target_name_idx", "inflictor_name_idx", "value"]
event_profile = combat[["match_id", "type"]].copy()
for col in profile_fields:
    event_profile[col + "_nulls"] = combat[col].isna().astype("int64")
event_profile = event_profile.groupby(["match_id", "type"], dropna=False).sum().reset_index()
event_counts = combat.groupby(["match_id", "type"], dropna=False).size().rename("rows").reset_index()
event_profile = event_counts.merge(event_profile, on=["match_id", "type"], validate="one_to_one")
EVENT_MATCH_ID = MATCH_ID  # Set None to show all matches.
display(event_profile if EVENT_MATCH_ID is None else event_profile.loc[event_profile["match_id"].eq(EVENT_MATCH_ID)])

,match_id,type,rows,attacker_name_idx_nulls,target_name_idx_nulls,inflictor_name_idx_nulls,value_nulls
0,9079737909,DOTA_COMBATLOG_ABILITY,271,0,40,0,271
1,9079737909,DOTA_COMBATLOG_ABILITY_TRIGGER,100,0,0,0,0
2,9079737909,DOTA_COMBATLOG_CRITICAL_DAMAGE,7,0,0,7,0
3,9079737909,DOTA_COMBATLOG_DAMAGE,10168,0,0,8598,0
4,9079737909,DOTA_COMBATLOG_DEATH,1442,0,0,1334,0
5,9079737909,DOTA_COMBATLOG_FIRST_BLOOD,1,1,1,1,1
6,9079737909,DOTA_COMBATLOG_GAME_STATE,10,10,10,10,0
7,9079737909,DOTA_COMBATLOG_GOLD,840,840,0,840,0
8,9079737909,DOTA_COMBATLOG_HEAL,2444,531,0,2167,0
9,9079737909,DOTA_COMBATLOG_ITEM,259,0,259,0,259


## 6. Unusual raw values and unresolved name references

Review distributions of player IDs, teams, slots, hero selections, game states, and lifecycle events.
These are raw values; team 1 or hero ID 0 can represent a non-playing entry or an earlier state.
A name-reference report distinguishes absent indexes from non-null indexes missing in the lookup.
All lookups below include `match_id`; they count references without changing event values.

In [ ]:
DISTRIBUTIONS = {
    "heroes": ["m_iPlayerID", "m_iTeamNum", "m_hReplicatingOtherHeroModel"],
    "player_resource": ["index", "m_vecPlayerData.m_iPlayerTeam", "m_vecPlayerTeamData.m_iTeamSlot",
                        "m_vecPlayerTeamData.m_nSelectedHeroID"],
    "gamerules": ["m_pGameRules.m_nGameState"],
    "wards": ["class", "event"], "item_entities": ["event"],
}
for table, columns in DISTRIBUTIONS.items():
    if table not in raw:
        continue
    part = raw[table].loc[raw[table]["match_id"].eq(MATCH_ID)]
    print(f"\n{table} | match {MATCH_ID}")
    display(part.groupby(columns, dropna=False).agg(rows=("match_id", "size"),
        first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index())

reference_rows = []
for field in ["attacker_name_idx", "target_name_idx", "inflictor_name_idx", "damage_source_name_idx", "target_source_name_idx"]:
    if field not in combat:
        continue
    refs = combat[["match_id", field]]
    lookup = raw["names"][["match_id", "idx"]].dropna().drop_duplicates().rename(columns={"idx": field})
    checked = refs.merge(lookup.assign(reference_exists=True), on=["match_id", field], how="left", validate="many_to_one")
    for match_id, part in checked.groupby("match_id"):
        reference_rows.append({"match_id": match_id, "reference": field,
            "absent_index_rows": int(part[field].isna().sum()),
            "unresolved_nonnull_rows": int((part[field].notna() & part["reference_exists"].isna()).sum())})
name_reference_profile = pd.DataFrame(reference_rows)
display(name_reference_profile)


heroes | match 9079737909


,m_iPlayerID,m_iTeamNum,m_hReplicatingOtherHeroModel,rows,first_tick,last_tick
0,0,2,16777215,1457,1560,45240
1,2,2,16777215,1457,1560,45240
2,4,2,16777215,1457,1560,45240
3,6,2,16777215,1457,1560,45240
4,8,2,16777215,1457,1560,45240
5,10,3,16777215,1457,1560,45240
6,12,3,16777215,1457,1560,45240
7,14,3,16777215,1457,1560,45240
8,16,3,16777215,1457,1560,45240
9,18,3,16777215,1457,1560,45240



player_resource | match 9079737909


,index,m_vecPlayerData.m_iPlayerTeam,m_vecPlayerTeamData.m_iTeamSlot,m_vecPlayerTeamData.m_nSelectedHeroID,rows,first_tick,last_tick
0,0,2,0,8,1508,30,45240
1,1,2,1,0,2,30,60
2,1,2,1,104,1506,90,45240
3,2,2,2,0,5,30,150
4,2,2,2,29,1503,180,45240
5,3,2,3,2,1508,30,45240
6,4,2,4,0,2,30,60
7,4,2,4,35,1506,90,45240
8,5,3,0,0,2,30,60
9,5,3,0,6,1506,90,45240



gamerules | match 9079737909


,m_pGameRules.m_nGameState,rows,first_tick,last_tick
0,2,5,30,150
1,3,30,180,1050
2,4,90,1560,4230
3,5,1222,4260,40890
4,6,120,40920,44490
5,7,25,44520,45240
6,8,16,1080,1530



wards | match 9079737909


,class,event,rows,first_tick,last_tick



item_entities | match 9079737909


,event,rows,first_tick,last_tick
0,created,357,14,40250
1,deleted,264,4648,40250


,match_id,reference,absent_index_rows,unresolved_nonnull_rows
0,9079737909,attacker_name_idx,3014,0
1,9079737994,attacker_name_idx,5012,0
2,9079738443,attacker_name_idx,1149,0
3,9079738574,attacker_name_idx,2379,0
4,9079740225,attacker_name_idx,413,0
5,9079740632,attacker_name_idx,2941,0
6,9079741152,attacker_name_idx,1680,0
7,9079743269,attacker_name_idx,893,0
8,9079745340,attacker_name_idx,1270,0
9,9079746862,attacker_name_idx,1049,0


## 7. Investigate specific raw rows

Choose a table, match, event type, raw player column/ID, tick range, or repeated-key columns.
Player numbers are used exactly as stored in the chosen table, without conversion. For example,
use `index=10` in `player_resource` to inspect the extra entry identified in match 9079741152.
Tick ranges are inclusive; repeated-key filtering is applied after the other filters.

In [ ]:
def inspect_rows(table, match_id, *, event_type=None, player_column=None, player_id=None,
                 tick_min=None, tick_max=None, repeated_keys=None, columns=None, limit=20):
    if table not in TABLES:
        raise ValueError(f"Unknown table: {table}")
    if match_id not in match_ids:
        raise ValueError(f"Match is not loaded: {match_id}")
    if limit < 1:
        raise ValueError("limit must be positive")
    if table not in raw:
        return pd.DataFrame()
    df = raw[table]
    part = df.loc[df["match_id"].eq(match_id)]
    needed = (["type"] if event_type is not None else []) + ([player_column] if player_column else [])
    needed += (["tick"] if tick_min is not None or tick_max is not None else [])
    needed += list(repeated_keys or []) + list(columns or [])
    missing = [c for c in needed if c not in df]
    if missing:
        raise ValueError(f"Columns not present in {table}: {missing}")
    if (player_column is None) != (player_id is None):
        raise ValueError("Provide both player_column and player_id, or neither.")
    if event_type is not None:
        part = part.loc[part["type"].eq(event_type).fillna(False)]
    if player_column is not None:
        part = part.loc[part[player_column].eq(player_id).fillna(False)]
    if tick_min is not None:
        part = part.loc[part["tick"].ge(tick_min).fillna(False)]
    if tick_max is not None:
        part = part.loc[part["tick"].le(tick_max).fillna(False)]
    if repeated_keys:
        part = part.loc[part.duplicated(["match_id", *repeated_keys], keep=False)]
    if "tick" in part:
        part = part.sort_values("tick", kind="stable")
    print(f"{len(part):,} matching raw rows; showing up to {limit}")
    return part.loc[:, columns].head(limit).copy() if columns is not None else part.head(limit).copy()

# Edit these controls, then rerun only this cell.
BROWSE_TABLE = "combatlog"
BROWSE_MATCH_ID = MATCH_ID
BROWSE_EVENT = "DOTA_COMBATLOG_DEATH"  # None shows all event types.
BROWSE_PLAYER_COLUMN = None  # For example: "index" or "m_iPlayerID".
BROWSE_PLAYER_ID = None
BROWSE_TICK_MIN = None
BROWSE_TICK_MAX = None
BROWSE_REPEATED_KEYS = None  # Example for combatlog: ["tick", "type", "target_name_idx"].
BROWSE_COLUMNS = None  # None shows all columns, or provide a list of raw column names.
display(inspect_rows(BROWSE_TABLE, BROWSE_MATCH_ID, event_type=BROWSE_EVENT,
    player_column=BROWSE_PLAYER_COLUMN, player_id=BROWSE_PLAYER_ID,
    tick_min=BROWSE_TICK_MIN, tick_max=BROWSE_TICK_MAX,
    repeated_keys=BROWSE_REPEATED_KEYS, columns=BROWSE_COLUMNS, limit=10))

1,442 matching raw rows; showing up to 10


,tick,type,timestamp,attacker_name_idx,target_name_idx,inflictor_name_idx,value,ability_level,ability_toggle_off,ability_toggle_on,armor_debuff_modifier,assist_players,at_night_time,attacker_has_scepter,attacker_hero,...,target_self,target_source_name_idx,target_team,timestamp_raw,total_unit_death_count,tracked_stat_id,ultimate_ability,unit_status_label,uses_charges,visible_dire,visible_radiant,will_reincarnate,xp_reason,xpm,match_id
155,4978,DOTA_COMBATLOG_DEATH,179.333344,49,47,<NA>,31,<NA>,None,None,<NA>,<NA>,False,False,False,...,False,47,3,179.333344,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
173,5196,DOTA_COMBATLOG_DEATH,186.600006,47,51,<NA>,16,<NA>,None,None,<NA>,<NA>,False,False,False,...,False,51,2,186.600006,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
185,5256,DOTA_COMBATLOG_DEATH,188.566681,9,47,<NA>,44,<NA>,None,None,<NA>,<NA>,False,False,True,...,False,47,3,188.566681,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
190,5294,DOTA_COMBATLOG_DEATH,189.866684,45,51,<NA>,4,<NA>,None,None,<NA>,<NA>,False,False,False,...,False,51,2,189.866684,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
195,5308,DOTA_COMBATLOG_DEATH,190.300003,1,47,<NA>,47,<NA>,None,None,<NA>,<NA>,False,False,True,...,False,47,3,190.300003,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
210,5360,DOTA_COMBATLOG_DEATH,192.066681,14,51,<NA>,18,<NA>,None,None,<NA>,<NA>,False,False,True,...,False,51,2,192.066681,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
212,5366,DOTA_COMBATLOG_DEATH,192.233337,49,47,<NA>,23,<NA>,None,None,<NA>,<NA>,False,False,False,...,False,47,3,192.233337,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
217,5398,DOTA_COMBATLOG_DEATH,193.333344,10,51,<NA>,7,<NA>,None,None,<NA>,<NA>,False,False,True,...,False,51,2,193.333344,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
226,5532,DOTA_COMBATLOG_DEATH,197.800003,14,51,<NA>,59,<NA>,None,None,<NA>,<NA>,False,False,True,...,False,51,2,197.800003,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909
231,5552,DOTA_COMBATLOG_DEATH,198.43335,49,45,<NA>,14,<NA>,None,None,<NA>,<NA>,False,False,False,...,False,45,3,198.43335,<NA>,<NA>,<NA>,<NA>,None,True,True,False,<NA>,<NA>,9079737909


## Record observations for the cleaning walkthrough

Record what the source contains, with match IDs, ticks, and sample values. Choose the cleaning
rule and validate its effects in [cleaning_walkthrough.ipynb](cleaning_walkthrough.ipynb).

| Observation | Match / raw columns / example ticks | Question to investigate |
|---|---|---|
| Add an observation | | |

The inspection notebook intentionally does not remap players, select final hero assignments,
convert game time or coordinates, aggregate damage/purchases, infer kill/assist credit, or build
final inventory. Those transformations belong in the walkthrough or production clean job.